In [1]:
import requests
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

Import API key from '.env' file

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

True

Ensure data folder exists

In [3]:
os.makedirs('../data/raw', exist_ok=True)

### Request data using API
If this takes too long, just download the dataset from kaggle (https://www.kaggle.com/datasets/dhanushnarayananr/credit-card-fraud)

In [12]:
url = "https://data.nayaone.com/credit_card_data"
batch_size = 10 # API rate limit per request
limit_size = None # max rows (1000000 rows ~16mins)
output_path = '../data/raw/credit_dataTEST.csv'

In [ ]:
headers = {
    'Accept-Profile': 'api',
    'sandpit-key': os.getenv('API_KEY')
}

def fetch_page(offset):
    response = requests.get(url, headers=headers, params={'offset': offset})
    return offset, response.json()

all_rows = []
offset = 0
active = True

max_workers = 30
executor = ThreadPoolExecutor(max_workers=max_workers)

pbar = tqdm(unit=" rows", desc="Fetching Data")

while active:
    # Build offsets for this wave
    offsets = [
        offset + i * batch_size
        for i in range(max_workers)
        if limit_size is None or offset + i * batch_size < limit_size
    ]

    if not offsets:
        break

    # Submit all tasks at once
    futures = [executor.submit(fetch_page, off) for off in offsets]

    # Wait for all tasks to finish (no as_completed)
    results = [f.result() for f in futures]

    # Sort by offset to preserve order
    results.sort(key=lambda x: x[0])

    # Process results
    for off, data in results:
        if not data:
            active = False
            break

        all_rows.extend(data)
        pbar.update(len(data))

    offset += max_workers * batch_size

pbar.close()

df = pd.DataFrame(all_rows)
df.to_csv(output_path, index=False)
print(f"{len(all_rows)} rows saved to {output_path}")

Fetching Data: 1000 rows [00:14, 68.63 rows/s]

1000 rows saved to ../data/raw/credit_data.csv


In [ ]:
headers = {
    'Accept-Profile': 'api',
    'sandpit-key': os.getenv('API_KEY')
}

all_rows = []
offset = 0

# create progress bar
pbar = tqdm(total=limit_size, unit=" rows", desc="Fetching Data")

while limit_size is None or offset < limit_size:
    response = requests.get(url, headers=headers, params={'offset': offset})
    
    # convert text into list of dictionaries
    data = response.json()
    
    if not data:
        break
        
    all_rows.extend(data)
    
    offset += batch_size
    
    pbar.update(batch_size)

pbar.close()

# convert list of dictionaries into table
df = pd.DataFrame(all_rows)

df.to_csv(output_path, index=False)
print (f'{len(all_rows)} rows saved to {output_path}')

In [ ]:
df